In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import PROCESSED_DATA_DIR, FIGURES_DIR
from src.plots import save_chart
from category_encoders import TargetEncoder

vs = {"width": 1200, "height": 700, "scale": 3, "renderer": "png"}
vs_box = {"width": 1200, "height": 1200, "scale": 3, "renderer": "png"}

In [ ]:
data = pd.read_feather(PROCESSED_DATA_DIR / 'train.feather')

# Типизация признаков (числовые, категориальные, временные) и их распределения


Датасет содержит числовые, категориальные и временные признаки:

- **Числовые:** возраст, доход, количество иждивенцев, баллы здоровья, предыдущие обращения, возраст автомобиля, кредитный рейтинг, срок страхования, сумма премии.
- **Категориальные:** пол, семейное положение, уровень образования, профессия, местоположение, тип полиса, обратная связь клиента, статус курения, частота упражнений, тип собственности.
- **Временные:** дата начала полиса.



In [ ]:
pd.DataFrame({
    "n_unique": data.nunique(),
    "dtype": data.dtypes
})

In [ ]:
def plot_boxplots_grid(data_: pd.DataFrame):
    """
    Plot box plots for numerical features in a grid
    """

    # Identify numerical columns
    numerical_columns = data_.select_dtypes(include=['float64', 'int64']).columns

    # Determine grid size based on the number of features
    num_features = len(numerical_columns)
    cols = 3  # Number of columns in the grid
    rows = -(-num_features // cols)  # Calculate rows, round up

    # Create subplots
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=numerical_columns)

    # Add box plots for each numerical column
    for idx, column in enumerate(numerical_columns):
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        fig.add_trace(
            go.Box(y=data_[column], name=column),
            row=row, col=col
        )

    # Update layout
    fig.update_layout(
        title="Box Plots for Numerical Features",
        height=300 * rows,  # Adjust height based on rows
        showlegend=False
    )

    return fig


fig_distribution = plot_boxplots_grid(data.drop(columns=["ID", "POLICY_START_DATE"]))
fig_distribution.show(**vs_box)

In [ ]:
def plot_barcharts_grid(data_: pd.DataFrame):
    """
    Function to plot bar charts in a grid for categorical features
    """
    # Identify categorical columns
    categorical_columns = data_.select_dtypes(include=['object', 'category']).columns

    # Determine grid size based on the number of features
    num_features = len(categorical_columns)
    cols = 3  # Number of columns in the grid
    rows = -(-num_features // cols)  # Calculate rows, round up

    # Create subplots
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=categorical_columns)

    # Add bar charts for each categorical column
    for idx, column in enumerate(categorical_columns):
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        value_counts = data_[column].fillna("NAN VALUE").value_counts()
        fig.add_trace(
            go.Bar(x=value_counts.index, y=value_counts.values, name=column),
            row=row, col=col
        )

    # Update layout
    fig.update_layout(
        title="Bar Charts for Categorical Features",
        height=300 * rows,  # Adjust height based on rows
        showlegend=False
    )

    return fig


fig_bars = plot_barcharts_grid(data.drop(columns=["ID", "POLICY_START_DATE"]))
fig_bars.show(**vs_box)

# Выявление аномальных значений

Данные выглядят очищенными или синтетическими:
- Категориальные данные распределены равномерно.
- Числовые данные практически не содержат выбросов, за исключением возможных выбросов в **PREVIOUS_CLAIMS**.
- Для **ANNUAL_INCOME** и **PREMIUM_AMOUNT** широкий правый хвост является нормальным, что соответствует классическому распределению таких величин.

# Анализ зависимостей между признаками

Матрица корреляции показывает, что переменные практически не связаны друг с другом. При условии, что мы использовали TargetEncoder для категориальных признаков, даже при этом методе кодирования корреляция между признаками остается низкой. Это хорошо, так как это означает, что признаки не являются линейно зависимыми и не содержат мультиколлинеарности.



In [ ]:
encoder: TargetEncoder = TargetEncoder()
encoder.fit(data, data.PREMIUM_AMOUNT)
data_enc = encoder.transform(data)

data_enc["POLICY_START_DATE"] = data["POLICY_START_DATE"]
data_enc.head(5)

In [ ]:
pd.DataFrame({
    "n_unique": data_enc.nunique(),
    "dtype": data_enc.dtypes
})

In [ ]:
corr_table = data_enc.corr()

fig_corr = px.imshow(corr_table, text_auto=".0%")
fig_corr = fig_corr.update_xaxes(nticks=20)
fig_corr = fig_corr.update_yaxes(nticks=30)

fig_corr.show(**vs)

# Анализ пропущенных значений

Доля пропущенных значений в датасете стабильна по месяцам. В следующих полях присутствуют пропущенные значения:

1. PREVIOUS_CLAIMS – 30% пропущенных значений. В этом поле также присутствуют нулевые значения, что может означать отсутствие предыдущих обращений. Можем заполнить пропуски нулями.
2. OCCUPATION – 30% пропущенных значений. Возможно, клиенты не указывают свою профессию. Таблица корреляции показывает, что это поле не связано с PREVIOUS_CLAIMS в плане пропущенных значений. Не можем заполнить пропуски модой, так как это приведет к смещению данных. Можем попробовать заполнить пропуски отдельной категорией.
3. CREDIT_SCORE – 11% пропущенных значений. Можем попробовать заполнить пропуски медианой или средним значением.
4. NUMBER_OF_DEPENDENTS – 9% пропущенных значений. Можем попробовать заполнить пропуски медианой или средним значением.
5. CUSTOMER_FEEDBACK – 6.4% пропущенных значений
6. HEALTH_SCORE – 6.1% пропущенных значений
7. ANNUAL_INCOME – 3.7% пропущенных значений
8. AGE – 1.5% пропущенных значений
9. MARITAL_STATUS – 1.5% пропущенных значений
10. VEHICLE_AGE – 6 пропусков, можем заполнить модой.
11. INSURANCE_DURATION – один пропуск, можем заполнить модой.


In [ ]:
missing_values = data.isnull().agg(["sum", "mean"]).T
missing_values.columns = ["n_missing", "p_missing"]
missing_values = missing_values.sort_values("p_missing", ascending=False)

missing_values

In [ ]:
data['MONTH'] = pd.to_datetime(data["POLICY_START_DATE"]).dt.to_period('M')
data['MONTH'] = data['MONTH'].dt.strftime('%Y-%m-%d')

# group by month and calc null share for each feature
features = missing_values[missing_values['p_missing'] > 0].index
data_nulls = data[features].isnull().astype(int)
data_nulls['MONTH'] = data['MONTH']

missing_values_month = data_nulls.groupby('MONTH').mean()
missing_values_month = missing_values_month.reset_index()

fig_missing = make_subplots(rows=1, cols=1)
for feature in features:
    fig_missing.add_trace(
        go.Scatter(x=missing_values_month['MONTH'], y=missing_values_month[feature], mode='lines', name=feature),
    )

fig_missing.update_layout(
    title="<b>Share of missing values by month",
    xaxis_title="Month",
    yaxis_title="<b>Share of missing values",
    height=600,
    showlegend=True
)

fig_missing.show(**vs)

In [ ]:
data_nulls.drop(columns='MONTH').corr()

# Определение важности признаков (корреляции с таргетом)

На основе корреляции с таргетом можно сделать следующие выводы (график выше):

1. Все переменные имеют около-нулевую корреляцию с таргетом


# Анализ возможных преобразований и генерации новых признаков



In [ ]:
from src.plots import feature_analysis

features = ['AGE', 'GENDER', 'ANNUAL_INCOME', 'MARITAL_STATUS',
            'NUMBER_OF_DEPENDENTS', 'EDUCATION_LEVEL', 'OCCUPATION', 'HEALTH_SCORE',
            'LOCATION', 'POLICY_TYPE', 'PREVIOUS_CLAIMS', 'VEHICLE_AGE',
            'CREDIT_SCORE', 'INSURANCE_DURATION', 'CUSTOMER_FEEDBACK',
            'SMOKING_STATUS', 'EXERCISE_FREQUENCY', 'PROPERTY_TYPE']

figures = {}

vs_wide = {"width": 1200, "height": 500, "scale": 3, "renderer": "png"}
for feat in features:
    res = feature_analysis(data, feat, 'PREMIUM_AMOUNT', 'POLICY_START_DATE')
    figure = res['fig']
    figure.update_layout(title=f"<b>{feat} analysis")
    figure.show(**vs_wide)
    figures[f"{feat}_analysis".upper()] = res
    # break


In [ ]:
charts_x_names = [
    (fig_distribution, 'NUMERICAL_FEATURES_DISTRIBUTION'),
    (fig_bars, 'CATEGORICAL_FEATURES_DISTRIBUTION'),
    (fig_corr, 'MUTUAL_CORRELATION_TABLE'),
    (fig_missing, 'MISSING_VALUES_BY_MONTH'),
]

for chart, name in charts_x_names:
    save_chart(chart, name, FIGURES_DIR / 'eda' / 'features_analysis', vs_box)